In [7]:
import pandas as pd
import json

split = 'train'


with open(f"../data/brexit/HS-Brexit_{'dev' if split=='val' else split}.json", 'r') as f:
    md_train = json.load(f)

    
df = pd.DataFrame.from_dict(md_train, orient='index')
df.reset_index(inplace=True)
df.rename(columns={'index': 'id'}, inplace=True)
# Split the "annotations" and "annotators" columns into separate lists
df['annotations'] = df['annotations'].apply(lambda x: x.split(','))
df['annotators'] = df['annotators'].apply(lambda x: x.split(','))

# Create a new DataFrame with each annotator and their annotation in a row
new_rows = []
for index, row in df.iterrows():
    for i, annotator in enumerate(row['annotators']):
        new_row = {
            'text': row['text'],
            'id': row['id'],
            'annotation task': row['annotation task'],
            'number of annotations': row['number of annotations'],
            'annotation': int(row['annotations'][i]),
            'annotator': annotator,
            'lang': row['lang'],
            'hard_label': row['hard_label'],
            'soft_label': row['soft_label'],
            'split': row['split'],
            'other_info': row['other_info']
        }
        new_rows.append(new_row)

# Create a new DataFrame with the transformed data
new_df = pd.DataFrame(new_rows)
new_df.head(20)


,text,id,annotation task,number of annotations,annotation,annotator,lang,hard_label,soft_label,split,other_info
0,<user> <user> I'm so glad about #Brexit.. My a...,1,hate speech detection,6,0,Ann1,en,0,"{'0': 1.0, '1': 0.0}",train,{'other annotations': {'aggressive language de...
1,<user> <user> I'm so glad about #Brexit.. My a...,1,hate speech detection,6,0,Ann2,en,0,"{'0': 1.0, '1': 0.0}",train,{'other annotations': {'aggressive language de...
2,<user> <user> I'm so glad about #Brexit.. My a...,1,hate speech detection,6,0,Ann3,en,0,"{'0': 1.0, '1': 0.0}",train,{'other annotations': {'aggressive language de...
3,<user> <user> I'm so glad about #Brexit.. My a...,1,hate speech detection,6,0,Ann4,en,0,"{'0': 1.0, '1': 0.0}",train,{'other annotations': {'aggressive language de...
4,<user> <user> I'm so glad about #Brexit.. My a...,1,hate speech detection,6,0,Ann5,en,0,"{'0': 1.0, '1': 0.0}",train,{'other annotations': {'aggressive language de...
5,<user> <user> I'm so glad about #Brexit.. My a...,1,hate speech detection,6,0,Ann6,en,0,"{'0': 1.0, '1': 0.0}",train,{'other annotations': {'aggressive language de...
6,RT <user>: There was more to #Brexit than immi...,2,hate speech detection,6,0,Ann1,en,0,"{'0': 1.0, '1': 0.0}",train,{'other annotations': {'aggressive language de...
7,RT <user>: There was more to #Brexit than immi...,2,hate speech detection,6,0,Ann2,en,0,"{'0': 1.0, '1': 0.0}",train,{'other annotations': {'aggressive language de...
8,RT <user>: There was more to #Brexit than immi...,2,hate speech detection,6,0,Ann3,en,0,"{'0': 1.0, '1': 0.0}",train,{'other annotations': {'aggressive language de...
9,RT <user>: There was more to #Brexit than immi...,2,hate speech detection,6,0,Ann4,en,0,"{'0': 1.0, '1': 0.0}",train,{'other annotations': {'aggressive language de...


In [8]:
import numpy as np
dis_df = new_df.groupby('id')['annotation'].value_counts().unstack().fillna(0).astype(int).reset_index()
dis_df['aggrement'] = (dis_df[0] - dis_df[1]).abs()
dis_df

annotation,id,0,1,aggrement
0,1,6,0,6
1,10,6,0,6
2,100,6,0,6
3,101,6,0,6
4,102,6,0,6
...,...,...,...,...
779,95,6,0,6
780,96,6,0,6
781,97,6,0,6
782,98,6,0,6


In [9]:
mv_df = new_df.groupby('id')['annotation'].value_counts().unstack().fillna(0).idxmax(axis=1).reset_index()
mv_df.rename(columns={0: 'mv'}, inplace=True)



In [10]:
joint_df = pd.merge(new_df, dis_df, on='id', how='left')
joint_df = joint_df[['text', 'id', 'annotation', 'annotator', 'aggrement']]
joint_df.rename(columns={'annotation': 'Hate'}, inplace=True)
joint_df = pd.merge(joint_df, mv_df, on='id', how='left')
joint_df['agree_mv'] = (joint_df['Hate'] == joint_df['mv']).astype(int)

In [11]:
import os

for ann in joint_df['annotator'].unique():
    
    data_path = f"../data/brexit/Hate/annotators/{ann}"
    os.makedirs(data_path, exist_ok=True)
    
    df_ann = joint_df[joint_df['annotator'] == ann]
    print(ann, df_ann['Hate'].value_counts())
    df_ann.to_csv(f"{data_path}/{split}.csv", index=False)

Ann1 Hate
0    749
1     35
Name: count, dtype: int64
Ann2 Hate
0    741
1     43
Name: count, dtype: int64
Ann3 Hate
0    735
1     49
Name: count, dtype: int64
Ann4 Hate
0    631
1    153
Name: count, dtype: int64
Ann5 Hate
0    587
1    197
Name: count, dtype: int64
Ann6 Hate
0    656
1    128
Name: count, dtype: int64
